### Links and Resources
| Concept | Description |
|---|---|
| [`explain`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html) | Read query plans |
| [Photon](https://www.databricks.com/product/photon) | Vectorized engine — only built-ins |
| [Catalyst](https://databricks.com/glossary/catalyst-optimizer) | Optimizes only built-ins / SQL UDFs |
| [Arrow batching](https://spark.apache.org/docs/latest/api/python/user_guide/sql/arrow_pandas.html) | What makes Pandas UDFs fast |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Performance Comparison: Built-in vs UDF vs Pandas UDF

Same logic, three ways. We'll compute `(amount * 1.21) + 0.5` rounded — a trivial transform — on 5M rows and time each approach. Then we'll read the query plans and explain *why* the numbers come out the way they do.

In [ ]:
import time

N = 5_000_000
df = spark.range(N).select(
    F.col("id").alias("amount"),
).repartition(8).cache()
df.count()  # materialize cache
print("rows:", N)

## A small timing helper

In [ ]:
def time_query(name, build_query, runs=3):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        build_query().write.format("noop").mode("overwrite").save()
        times.append(time.perf_counter() - t0)
    best = min(times)
    print(f"{name:30s} best of {runs}: {best*1000:7.1f} ms")
    return best

## Three implementations of the same logic

In [ ]:
# 1. Pure built-in
def built_in_query():
    return df.select(F.round(F.col("amount") * 1.21 + 0.5, 2).alias("v"))

# 2. Plain Python UDF
@F.udf("double")
def py_udf(x):
    return None if x is None else round(x * 1.21 + 0.5, 2)

def py_udf_query():
    return df.select(py_udf("amount").alias("v"))

# 3. Pandas UDF
import pandas as pd
@F.pandas_udf("double")
def pd_udf(x: pd.Series) -> pd.Series:
    return (x * 1.21 + 0.5).round(2)

def pd_udf_query():
    return df.select(pd_udf("amount").alias("v"))

## Run the benchmark

In [ ]:
t1 = time_query("built-in",   built_in_query)
t2 = time_query("python UDF",  py_udf_query)
t3 = time_query("pandas UDF",  pd_udf_query)

print()
print(f"python UDF / built-in: {t2/t1:.1f}x slower")
print(f"pandas UDF / built-in: {t3/t1:.1f}x slower")
print(f"python UDF / pandas UDF: {t2/t3:.1f}x slower")

## Why? Read the plan.

`explain(mode="formatted")` gives a clean view of stages and the physical operators.

In [ ]:
print("=" * 60, "BUILT-IN")
built_in_query().explain(mode="formatted")

print("=" * 60, "PYTHON UDF")
py_udf_query().explain(mode="formatted")

print("=" * 60, "PANDAS UDF")
pd_udf_query().explain(mode="formatted")

## What to look for in those plans

- **Built-in** — single `Project` operator. Catalyst can fuse it with neighbours, push it into Parquet readers, and Photon can run it natively.
- **Python UDF** — a `BatchEvalPython` operator. Each row makes a JVM↔Python round trip. Photon can't run anything past this point.
- **Pandas UDF** — an `ArrowEvalPython` operator. Rows are batched into Arrow record batches; Python runs once per batch (10k rows at default settings).

## The Photon dimension

When the cluster has **Photon** enabled (most current Databricks SQL warehouses, many DBR clusters), the gap between built-ins and UDFs widens further. Photon is a vectorized C++ engine that *only operates on built-ins*. The moment a Python UDF appears in the plan, Photon hands the data back to the JVM, runs the Python step, and tries to resume Photon afterwards. That handoff is expensive in itself.

Look for `PhotonProject` vs `Project` in the plan — Photon stages will be tagged.

## Practical decision tree

```
Can I express this with F.* / SQL?      ── yes ─→ Built-in (notebooks 02-10)
                │ no
                ▼
Can it be vectorized w/ pandas/numpy?  ── yes ─→ Pandas UDF (notebook 12)
                │ no
                ▼
Is the row volume small (< 100k)?       ── yes ─→ Plain Python UDF is fine
                │ no
                ▼
Reconsider — pull data out via collect, do it in pandas, push back.
```

## Other levers worth measuring

- **Arrow-optimized Python UDFs**: `@F.udf(returnType, useArrow=True)` — frequently 2× the plain version, with no code change.
- **Batch size**: `spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", n)` — bigger batches amortize Python overhead but cost worker memory.
- **`repartition(n)`** — too few partitions and one slow Python worker becomes the bottleneck.

In [ ]:
df.unpersist()

## Takeaways

- Built-ins are the reference point. Anything else is measured *relative to them*.
- Python UDFs add a Python worker per row — typically 10–50× slower than equivalent built-ins.
- Pandas UDFs add a Python worker per *batch* via Arrow — typically 1.5–3× a built-in, an order of magnitude faster than a plain UDF.
- `explain(mode="formatted")` reveals which path you're on: `Project` (good), `ArrowEvalPython` (acceptable), `BatchEvalPython` (slow).
- Photon doubles the built-in advantage. Any Python step disables Photon for that stage.